# 01_da_agreement_template

Standalone **Data Agreement Intake / Usage Boundary** notebook. It writes one append-only row per agreement version to `METADATA_DATA_AGREEMENT`. Governance classification and review remain in `04_gov_*`.


In [ ]:
%run 00_env_config


In [ ]:
from IPython.display import clear_output
from fabricops_kit import (
    collect_agreement_metadata,
    commit_agreement_metadata,
    create_agreement_form,
    load_agreements,
    read_agreement_form,
    setup_data_agreement_tables,
)


## One-time metadata setup

This creates empty agreement and steward tables by configured OneLake path. Populate `METADATA_DATA_STEWARD` with real active steward profiles before rendering the intake form; no fake people are seeded.


In [ ]:
setup_data_agreement_tables(spark=spark, config=CONFIG, env=ENV)


In [ ]:
agreement_form = create_agreement_form(spark=spark, config=CONFIG, env=ENV)


In [ ]:
def on_commit_clicked(_):
    with agreement_form["output"]:
        clear_output()
        try:
            latest = load_agreements(CONFIG, ENV, spark_session=spark, missing_ok=True)
            selected = agreement_form["existing_agreement"].value if agreement_form["mode"].value == "Update Existing Agreement" else None
            if agreement_form["mode"].value == "Update Existing Agreement" and not selected:
                raise ValueError("Update mode selected, but no existing agreement was chosen.")
            metadata = collect_agreement_metadata(widget_values=read_agreement_form(agreement_form), mode="update" if selected else "create", existing_rows=latest, selected_agreement=selected, config=CONFIG, env=ENV)
            summary = commit_agreement_metadata(spark=spark, config=CONFIG, env=ENV, agreement_metadata=metadata)
            print("Data agreement committed successfully.")
            print(summary)
        except Exception as exc:
            print("Commit failed.")
            print(str(exc))

agreement_form["commit_button"].on_click(on_commit_clicked)
